In [43]:
import pandas as pd
import numpy as np

sales = pd.read_csv('../data/processed/sales.csv')
np.random.seed(0)

In [44]:
sales['Date'] = pd.to_datetime(sales['Date'])

In [45]:
print(sales['Date'].dtype)

datetime64[us]


In [46]:
sales = sales.sort_values('Date').reset_index(drop=True)

In [47]:
# Lag features cho Revenue
for lag in [1, 2, 3, 7, 14, 30]:
    sales[f'revenue_lag_{lag}'] = sales['Revenue'].shift(lag)
# Lag features cho COGS
for lag in [1, 7, 30]:
    sales[f'cogs_lag_{lag}'] = sales['COGS'].shift(lag)

In [48]:
sales.head()

,Date,Revenue,COGS,is_holiday,log_Revenue,log_COGS,revenue_lag_1,revenue_lag_2,revenue_lag_3,revenue_lag_7,revenue_lag_14,revenue_lag_30,cogs_lag_1,cogs_lag_7,cogs_lag_30
0,2012-07-04,5123547.94,3982991.19,0,15.449358,15.197544,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-07-05,2751773.45,2150580.23,0,14.827757,14.581249,5123547.94,NaN,NaN,NaN,NaN,NaN,3982991.19,NaN,NaN
2,2012-07-06,3054029.42,2517632.84,0,14.931973,14.738830,2751773.45,5123547.94,NaN,NaN,NaN,NaN,2150580.23,NaN,NaN
3,2012-07-07,2667930.94,2108246.62,0,14.796814,14.561368,3054029.42,2751773.45,5123547.94,NaN,NaN,NaN,2517632.84,NaN,NaN
4,2012-07-08,2360851.90,1808622.79,0,14.674534,14.408077,2667930.94,3054029.42,2751773.45,NaN,NaN,NaN,2108246.62,NaN,NaN


In [49]:
# rolling mean
for w in [7, 14, 30]:
    sales[f'rev_roll{w}_mean'] = sales['Revenue'].shift(1).rolling(w).mean()
    if w in [7, 30]: 
        sales[f'rev_roll{w}_std'] = sales['Revenue'].shift(1).rolling(w).std()
        sales[f'rev_roll{w}_max'] = sales['Revenue'].shift(1).rolling(w).max()

In [50]:
# calendar 
sales['day_of_week'] = sales['Date'].dt.dayofweek 
sales['month'] = sales['Date'].dt.month
sales['day_of_month'] = sales['Date'].dt.day
sales['week_of_year'] = sales['Date'].dt.isocalendar().week.astype(int)
sales['quarter'] = sales['Date'].dt.quarter

sales['is_weekend'] = sales['day_of_week'].isin([5, 6]).astype(int)

fixed_holidays = []

for year in range(2012, 2026):
    fixed_holidays.extend([f'{year}-01-01', f'{year}-04-30', f'{year}-05-01', f'{year}-09-02'])
# gio to hung vuong
gioto_dates = [
    '2012-03-31', '2013-04-19', '2014-04-09', '2015-04-28', '2016-04-16', '2017-04-06', 
    '2018-04-25', '2019-04-14', '2020-04-02', '2021-04-21', '2022-04-10', '2023-04-29', 
    '2024-04-18', '2025-04-07'
]

all_holidays = pd.to_datetime(fixed_holidays + gioto_dates)

sales['is_holiday'] = sales['Date'].apply(lambda x: 1 if min(abs((x - h).days) for h in all_holidays) <= 3 else 0)

# Tet
tet_dates = pd.to_datetime([
    '2012-01-23', '2013-02-10', '2014-01-31', '2015-02-19', '2016-02-08', '2017-01-28', 
    '2018-02-16', '2019-02-05', '2020-01-25', '2021-02-12', '2022-02-01', '2023-01-22', 
    '2024-02-10', '2025-01-29'
])

sales['days_to_tet'] = sales['Date'].apply(lambda x: min(abs((x - t).days) for t in tet_dates)).clip(upper=30)

def check_is_tet(date_val):
    for t in tet_dates:
        diff = (date_val - t).days
        if -3 <= diff <= 7: 
            return 1
    return 0

sales['is_tet'] = sales['Date'].apply(check_is_tet)

In [51]:
# promo - khuyen mai
promotions = pd.read_csv('../data/processed/promotions.csv')

promotions['start_date'] = pd.to_datetime(promotions['start_date'], format="%Y-%m-%d")
promotions['end_date'] = pd.to_datetime(promotions['end_date'], format="%Y-%m-%d")

sales['promo_count'] = 0
for _, row in promotions.iterrows():
    mask = (sales['Date'] >= row['start_date']) & (sales['Date'] <= row['end_date'])
    sales.loc[mask, 'promo_count'] += 1
sales['promo_active'] = (sales['promo_count'] > 0).astype(int)


In [52]:
# web_traffic
web_traffic = pd.read_csv('../data/processed/web_traffic.csv')
web_traffic['date'] = pd.to_datetime(web_traffic['date'])

daily_web = web_traffic.groupby('date')['sessions'].sum().reset_index()
sales = sales.merge(daily_web, left_on='Date', right_on='date', how='left').drop(columns=['date'])

sales['sessions_lag7'] = sales['sessions'].shift(7)
sales['sessions_lag14'] = sales['sessions'].shift(14)
sales['sessions_roll7_mean'] = sales['sessions'].shift(1).rolling(7).mean()

In [53]:
print(sales.shape)

print("NaN head(30):")
missing_check = sales[['revenue_lag_30', 'cogs_lag_30', 'sessions_lag14']].isna().sum()
print(missing_check)

sales.to_csv('../data/processed/features_train.csv', index=False)

(3833, 36)
NaN head(30):
revenue_lag_30     30
cogs_lag_30        30
sessions_lag14    195
dtype: int64


In [54]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

features_train = pd.read_csv('../data/processed/features_train.csv')
features_train['Date'] = pd.to_datetime(features_train['Date'])

features_train = features_train.dropna().reset_index(drop=True)

In [55]:
# Features X
features = [col for col in features_train.columns if col not in ['Date', 'Revenue', 'COGS']]
X = features_train[features]
y_rev = features_train['Revenue']
y_cogs = features_train['COGS']

# TimeSeriesSplit(n_splits=5)
tscv = TimeSeriesSplit(n_splits=5)

for train_index, val_index in tscv.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train_rev, y_val_rev = y_rev.iloc[train_index], y_rev.iloc[val_index]
    y_train_cogs, y_val_cogs = y_cogs.iloc[train_index], y_cogs.iloc[val_index]

print(f"Train: {X_train.shape}\nVal: {X_val.shape}")

Train: (3032, 33)
Val: (606, 33)


In [56]:
# Train XGBoost
xgb_rev = xgb.XGBRegressor(
    n_estimators=500, 
    learning_rate=0.05, 
    max_depth=6, 
    random_state=42, 
    early_stopping_rounds=50
)
xgb_rev.fit(
    X_train, y_train_rev, 
    eval_set=[(X_val, y_val_rev)], 
    verbose=100
)

xgb_cogs = xgb.XGBRegressor(
    n_estimators=500, 
    learning_rate=0.05, 
    max_depth=6, 
    random_state=42,
    early_stopping_rounds=50
)
xgb_cogs.fit(
    X_train, y_train_cogs, 
    eval_set=[(X_val, y_val_cogs)], 
    verbose=100
)

[0]	validation_0-rmse:2107991.29960
[100]	validation_0-rmse:21395.25046
[187]	validation_0-rmse:19391.81151
[0]	validation_0-rmse:1752010.79083
[100]	validation_0-rmse:14208.34324
[183]	validation_0-rmse:10800.61923


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [57]:
# MAE, RMSE, R^2 tai X_val
def calc_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"[{name}] MAE: {mae:,.2f} | RMSE: {rmse:,.2f} | R²: {r2:.4f}")

pred_val_rev_xgb = xgb_rev.predict(X_val)
pred_val_cogs_xgb = xgb_cogs.predict(X_val)
calc_metrics(y_val_rev, pred_val_rev_xgb, "XGBoost Revenue")
calc_metrics(y_val_cogs, pred_val_cogs_xgb, "XGBoost COGS")

[XGBoost Revenue] MAE: 9,895.31 | RMSE: 19,130.56 | R²: 0.9999
[XGBoost COGS] MAE: 7,505.91 | RMSE: 10,516.94 | R²: 0.9999


In [58]:
# features (test period)
sample_sub = pd.read_csv('../data/processed/sample_submission.csv')
sample_sub['Date'] = pd.to_datetime(sample_sub['Date'])
last_30_train = features_train.tail(30).copy()
test_ft_mock = sample_sub.copy()
concat_ft = pd.concat([last_30_train, test_ft_mock], ignore_index=True)

for col in features:
    if col not in concat_ft.columns:
        concat_ft[col] = np.nan 
X_test = concat_ft[concat_ft['Date'] >= '2023-01-01'][features]

In [59]:
# Predict Revenue, COGS (548 ngày test)
test_preds_rev = xgb_rev.predict(X_test)
test_preds_cogs = xgb_cogs.predict(X_test)

test_preds_rev = np.round(np.clip(test_preds_rev, 0, None), 2)
test_preds_cogs = np.round(np.clip(test_preds_cogs, 0, None), 2)

submission = sample_sub.copy()
submission['Revenue'] = test_preds_rev
submission['COGS'] = test_preds_cogs

print(f"check line number (548): {len(submission)}")
assert len(submission) == 548, "ERROR"

check line number (548): 548


In [60]:
submission.to_csv('../submissions/submission_v1.csv', index=False)
print("Saved successfully: submissions/submission_v1.csv")

Saved successfully: submissions/submission_v1.csv
